In [2]:
import time
import pandas as pd 
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import emoji, re, string

import nltk
from nltk.corpus import stopwords
import spacy

from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import BernoulliNB, MultinomialNB
from sklearn.svm import LinearSVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import SGDClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.neural_network import MLPClassifier


from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, roc_curve
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler
from transformers import AutoTokenizer, AutoModel
import torch
import numpy as np

import warnings
warnings.filterwarnings("ignore")



d:\Arquivos_Acer\Documents\UFC\PrejudicePT-br-main\ambiente_prejudice\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
df = pd.read_csv('../../Datasets/Correto_whatsapp_rotulado_revisado.csv')
df.head()

,text_content_anonymous,preconceito
0,Show :clapping_hands_light_skin_tone::clapping...,0
1,Mais uma vez o PT= Partido das Trevas apela ao...,1
2,Se não puder avise só para não quebrar a vigíl...,0
3,E a esquerda tenta impedir o trabalho,0
4,Deus perdoa porque eles Não sabem que fazem e ...,0


In [3]:
df.shape

(3000, 2)

In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3000 entries, 0 to 2999
Data columns (total 2 columns):
 #   Column                  Non-Null Count  Dtype 
---  ------                  --------------  ----- 
 0   text_content_anonymous  2999 non-null   object
 1   preconceito             3000 non-null   int64 
dtypes: int64(1), object(1)
memory usage: 47.0+ KB


In [4]:
mensagens_coluna = 'text_content_anonymous'  
df[mensagens_coluna] = df[mensagens_coluna].astype(str) # garantindo que todos os valores são strings

# Tokenizar mensagens (simples divisão por espaços)
df['num_tokens'] = df[mensagens_coluna].apply(lambda x: len(x.split()))

## Processamento do texto

In [5]:
unicode_emoji = {}
for key, value in emoji.EMOJI_DATA.items():
    try:
        unicode_emoji[key] = value['pt']
    except:
        pass

#emojis and punctuation
emojis_list = list(unicode_emoji)
punct = list(string.punctuation)
emojis_punct = emojis_list + punct

def processEmojisPunctuation(text, remove_punct = True):
    '''
    Put spaces between emojis. Removes punctuation.
    '''
    #get all unique chars
    chars = set(text)
    #for each unique char in text, do:
    for c in chars:
        #remove punctuation
        if remove_punct:
            if c in emojis_list:
                text = text.replace(c, ' ' + c + ' ')
            if c in punct:
                text = text.replace(c, ' ')

        #put spaces between punctuation
        else:
            if c in emojis_punct:
                text = text.replace(c, ' ' + c + ' ')          

    text = text.replace('  ', ' ')
    return text

#stop words removal
stop_words = list(stopwords.words('portuguese'))
new_stopwords = ['aí','pra','vão','vou','onde','lá','aqui',
                 'tá','pode','pois','so','deu','agora','todo',
                 'nao','ja','vc', 'bom', 'ai','kkk','kkkk','ta', 'voce', 'alguem', 'ne', 'pq',
                 'cara','to','mim','la','vcs','tbm', 'tudo']
stop_words = stop_words + new_stopwords
final_stop_words = []
for sw in stop_words:
    sw = ' '+ sw + ' '
    final_stop_words.append(sw)

def removeStopwords(text):
    for sw in final_stop_words:
        text = text.replace(sw,' ')
    text = text.replace('  ',' ')
    return text

#lemmatization
nlp = spacy.load('pt_core_news_sm')
def lemmatization(text):
    doc = nlp(text)
    for token in doc:
        if token.text != token.lemma_:
            text = text.replace(token.text, token.lemma_)
    return text


def domainUrl(text):
    '''
    Substitutes an URL in a text for the domain of this URL
    Input: an string
    Output: the string with the modified URL
    '''    
    if 'http' in text:
        re_url = '[^\s]*https*://[^\s]*'
        matches = re.findall(re_url, text, flags=re.IGNORECASE)
        for m in matches:
            domain = m.split('//')
            domain = domain[1].split('/')[0]
            text = re.sub(re_url, domain, text, 1)
        return text
    else:
        return text 

def preprocess(text):
    text = text.lower().strip()
    text = domainUrl(text)
    text = processEmojisPunctuation(text)
    text = removeStopwords(text)
    text = lemmatization(text)
    return text


## Carregando dicionário 

In [6]:
# Função para carregar o dicionário
def carregar_dicionario_personalizado(dic_path):
    categorias = {}
    lexicon = {}
    dentro_das_categorias = False

    with open(dic_path, 'r', encoding='utf-8') as file:
        for linha in file:
            linha = linha.strip()
            
            # Detecta a seção de categorias delimitada por '%'
            if linha == '%':
                dentro_das_categorias = not dentro_das_categorias
                continue
            
            # Lê as categorias personalizadas
            if dentro_das_categorias:
                codigo, categoria = linha.split()
                categorias[codigo] = categoria
            else:
                # Lê as palavras e suas categorias
                partes = linha.split("\t")
                palavra = partes[0]
                categoria_ids = partes[1:]
                lexicon[palavra] = [categorias[codigo] for codigo in categoria_ids if codigo in categorias]
    
    return lexicon, list(categorias.values())

In [7]:
# Tokenização simples
def tokenize(text):
    tokens = []
    for match in re.finditer(r"\w+", text, re.UNICODE):
        tokens.append(match.group(0).lower())
    return tokens

# Função para incluir palavras do dicionário como colunas
def adicionar_palavras_como_features(df, lexicon):
    # Criar DataFrame temporário com todas as palavras inicializadas com 0
    new_columns = pd.DataFrame(0, index=df.index, columns=list(lexicon.keys()))
    
    # Concatenar com o DataFrame original de uma vez
    df = pd.concat([df, new_columns], axis=1)
    
    # Contar ocorrências de cada palavra no texto
    for i, texto in df['text'].items():
        tokens = tokenize(texto)
        for token in tokens:
            if token in lexicon:
                df.at[i, token] += 1
                
    return df

# Caminho do arquivo .dic personalizado
dic_path = '../../Dicionário/v2_SocialLIWC_formatado_ordenado.dic'
lexicon, category_names = carregar_dicionario_personalizado(dic_path)

In [8]:
# === Renomeando colunas ===
df = df.rename(columns={
    "text_content_anonymous": "text",
    "preconceito": "label"
})

df["label"] = df["label"].astype(int)


In [10]:
df.head()

,text,label,num_tokens
0,Show :clapping_hands_light_skin_tone::clapping...,0,5
1,Mais uma vez o PT= Partido das Trevas apela ao...,1,43
2,Se não puder avise só para não quebrar a vigíl...,0,123
3,E a esquerda tenta impedir o trabalho,0,7
4,Deus perdoa porque eles Não sabem que fazem e ...,0,15


In [9]:
# Função de pré-processamento
def preprocess_data(df, experiment):
    if 'processed' in experiment:
        print("Pré-processamento ativado.")
        pro_texts = [preprocess(t) for t in df['text']]
    else:
        print("Sem pré-processamento.")
        pro_texts = [processEmojisPunctuation(t.lower(), remove_punct=False) for t in df['text']]
    return pro_texts


In [10]:
classifiers = {
    "LogisticRegression": LogisticRegression(max_iter=1000),
    "BernoulliNB": BernoulliNB(),
    "MultinomialNB": MultinomialNB(),  # cuidado: exige features não-negativas
    "LinearSVC": LinearSVC(dual="auto", max_iter=10000, random_state=42),
    "KNN": KNeighborsClassifier(),
    "SGDClassifier": SGDClassifier(),
    "RandomForest": RandomForestClassifier(),
    "GradientBoosting": GradientBoostingClassifier(),
    "MLP": MLPClassifier(max_iter=300)
}

## Modelos Pré treinados - Sem pré processamento

In [13]:
# === Pré-processamento dinâmico ===
df["text"] = preprocess_data(df, experiment="raw")  
df.head()

Sem pré-processamento.


,text,label,num_tokens
0,show : clapping _ hands _ light _ skin _ tone ...,0,5
1,mais uma vez o pt = partido das trevas apela a...,1,43
2,se não puder avise só para não quebrar a vigíl...,0,123
3,e a esquerda tenta impedir o trabalho,0,7
4,deus perdoa porque eles não sabem que fazem e ...,0,15


In [14]:
df = adicionar_palavras_como_features(df, lexicon)
df.head()

,text,label,num_tokens,aberração,aberração*,aberrações,aleij*,aleija,aleijada,aleijadas,...,progressista,progressistinha,reaça,revolucionário,socialista,sociopata,tucanhalha,vermelhinho,xenofóbico,zumbi
0,show : clapping _ hands _ light _ skin _ tone ...,0,5,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,mais uma vez o pt = partido das trevas apela a...,1,43,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,se não puder avise só para não quebrar a vigíl...,0,123,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,e a esquerda tenta impedir o trabalho,0,7,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,deus perdoa porque eles não sabem que fazem e ...,0,15,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


### melll-uff/bertweetbr

In [15]:

# === Configurações ===

model_name = "melll-uff/bertweetbr"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModel.from_pretrained(
    model_name,
    use_safetensors=True # não usa toach.load
    )
model.eval()  # modo avaliação

def get_embeddings(texts, tokenizer, model, device="cpu", pooling="cls_last4"):
    inputs = tokenizer(
        texts, 
        padding=True, 
        truncation=True, 
        max_length=128, 
        return_tensors="pt"
    )
    inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = model(**inputs, output_hidden_states=True)
        hidden_states = outputs.hidden_states  # lista de camadas
        last_hidden = outputs.last_hidden_state  # [batch, seq_len, hidden]

        if pooling == "cls":
            embeddings = last_hidden[:, 0, :]

        elif pooling == "mean":
            embeddings = last_hidden.mean(dim=1)

        elif pooling == "max":
            embeddings = last_hidden.max(dim=1).values

        elif pooling == "mean_max":
            mean_emb = last_hidden.mean(dim=1)
            max_emb = last_hidden.max(dim=1).values
            embeddings = torch.cat([mean_emb, max_emb], dim=1)

        elif pooling == "cls_last4":
            # média das últimas 4 camadas do CLS
            last_four = hidden_states[-4:]
            stacked = torch.stack(last_four, dim=0)  # [4, batch, seq_len, hidden]
            embeddings = torch.mean(stacked, dim=0)[:, 0, :]

        else:
            raise ValueError("Pooling deve ser 'cls', 'mean', 'max', 'mean_max' ou 'cls_last4'")

    return embeddings.cpu().numpy()

'''def get_embeddings(texts, tokenizer, model, device="cpu", pooling="cls"):
    inputs = tokenizer(texts, padding=True, truncation=True, max_length=128, return_tensors="pt")
    inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = model(**inputs)
        last_hidden = outputs.last_hidden_state  # [batch, seq_len, hidden]

        if pooling == "cls":
            embeddings = last_hidden[:, 0, :]  # pega [CLS]
        elif pooling == "mean":
            embeddings = last_hidden.mean(dim=1)  # média dos tokens
        else:
            raise ValueError("Pooling deve ser 'cls' ou 'mean'")

    return embeddings.cpu().numpy()
'''

texts = df["text"].tolist()
labels = df["label"].values

# Usar palavras do dicionário
X_words = df[list(lexicon.keys())]
vocab_size_words = X_words.shape[1]

# Normalizar palavras 
scaler = StandardScaler()
X_words_scaled = scaler.fit_transform(X_words)


X_embeddings = get_embeddings(texts, tokenizer, model, pooling="cls_last4")

# Concatenar embeddings + palavras
X_full = np.hstack([X_embeddings, X_words_scaled])
y = labels



skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
results = []

experiments_start_time = time.time()

for clf_name, clf in classifiers.items():
    print(f"\n=== {clf_name} ===")
    fold_metrics = []
    
    
    for train_idx, test_idx in skf.split(X_full, y):
        X_train, X_test = X_full[train_idx], X_full[test_idx]
        y_train, y_test = y[train_idx], y[test_idx]
        
        # Ajuste especial para MultinomialNB (precisa de entradas positivas)
        if clf_name == "MultinomialNB":
            from sklearn.preprocessing import MinMaxScaler
            scaler = MinMaxScaler()
            X_train = scaler.fit_transform(X_train)
            X_test = scaler.transform(X_test)
        
        clf.fit(X_train, y_train)
        y_pred = clf.predict(X_test)
        
        # Probabilidades ou scores para AUC
        if hasattr(clf, "predict_proba"):
            y_scores = clf.predict_proba(X_test)[:, 1]
        elif hasattr(clf, "decision_function"):
            y_scores = clf.decision_function(X_test)
        else:
            y_scores = None  # se o modelo não tiver, pula AUC

        # Algumas métricas
        acc = accuracy_score(y_test, y_pred)
        prec = precision_score(y_test, y_pred)
        rec = recall_score(y_test, y_pred)
        f1 = f1_score(y_test, y_pred)
        
        if y_scores is not None:
            auc = roc_auc_score(y_test, y_scores)
        else:
            auc = np.nan

        fold_metrics.append([acc, prec, rec, f1, auc])
    
    
    fold_metrics = np.array(fold_metrics, dtype=np.float64)
    mean_metrics = np.nanmean(fold_metrics, axis=0)  # ignora NaN se algum modelo não tiver AUC
    std_metrics = np.nanstd(fold_metrics, axis=0)
    vocab_size = X_full.shape[1]

    results.append([
        clf_name,
        *mean_metrics,
        *std_metrics,
        vocab_size
    ])

# Tempo total do experimento
experiment_time = time.time() - experiments_start_time

# Organiza resultados em DataFrame
results_df = pd.DataFrame(
    results, 
    columns=[
        "Modelo", 
        "Accuracy_mean", "Precision_mean", "Recall_mean", "F1_mean", "AUC_mean",
        "Accuracy_std", "Precision_std", "Recall_std", "F1_std", "AUC_std",
        "Vocab_size_combined"
    ]
)

results_df['total_experiment_time_sec'] = experiment_time

print(results_df)
results_df.to_csv('./resultados/results_melll_uff_Sempre_processamento.csv', index=False)

Some weights of RobertaModel were not initialized from the model checkpoint at melll-uff/bertweetbr and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.



=== LogisticRegression ===

=== BernoulliNB ===

=== MultinomialNB ===

=== LinearSVC ===

=== KNN ===


  File "d:\Arquivos_Acer\Documents\UFC\PrejudicePT-br-main\ambiente_prejudice\lib\site-packages\joblib\externals\loky\backend\context.py", line 257, in _count_physical_cores
    cpu_info = subprocess.run(
  File "C:\Users\Melissa Felipe\AppData\Local\Programs\Python\Python310\lib\subprocess.py", line 503, in run
    with Popen(*popenargs, **kwargs) as process:
  File "C:\Users\Melissa Felipe\AppData\Local\Programs\Python\Python310\lib\subprocess.py", line 971, in __init__
    self._execute_child(args, executable, preexec_fn, close_fds,
  File "C:\Users\Melissa Felipe\AppData\Local\Programs\Python\Python310\lib\subprocess.py", line 1456, in _execute_child
    hp, ht, pid, tid = _winapi.CreateProcess(executable, args,



=== SGDClassifier ===

=== RandomForest ===

=== GradientBoosting ===

=== MLP ===
               Modelo  Accuracy_mean  Precision_mean  Recall_mean   F1_mean  \
0  LogisticRegression       0.866333        0.905029     0.818667  0.859519   
1         BernoulliNB       0.601333        0.617806     0.529333  0.570022   
2       MultinomialNB       0.730667        0.739583     0.711333  0.725146   
3           LinearSVC       0.873667        0.899051     0.842000  0.869530   
4                 KNN       0.794000        0.812149     0.765333  0.787903   
5       SGDClassifier       0.840333        0.826172     0.863333  0.843941   
6        RandomForest       0.793667        0.795089     0.791333  0.793139   
7    GradientBoosting       0.833333        0.846361     0.814667  0.830131   
8                 MLP       0.856000        0.858435     0.852667  0.855525   

   AUC_mean  Accuracy_std  Precision_std  Recall_std    F1_std   AUC_std  \
0  0.931187      0.007483       0.004039    0.019

### FpOliveira/tupi-bert-base-portuguese-cased

In [16]:
# === Configurações ===

model_name = "FpOliveira/tupi-bert-base-portuguese-cased"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModel.from_pretrained(
    model_name,
    use_safetensors=True # não usa toach.load
    )
model.eval()  # modo avaliação

def get_embeddings(texts, tokenizer, model, device="cpu", pooling="cls_last4"):
    inputs = tokenizer(
        texts, 
        padding=True, 
        truncation=True, 
        max_length=128, 
        return_tensors="pt"
    )
    inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = model(**inputs, output_hidden_states=True)
        hidden_states = outputs.hidden_states  # lista de camadas
        last_hidden = outputs.last_hidden_state  # [batch, seq_len, hidden]

        if pooling == "cls":
            embeddings = last_hidden[:, 0, :]

        elif pooling == "mean":
            embeddings = last_hidden.mean(dim=1)

        elif pooling == "max":
            embeddings = last_hidden.max(dim=1).values

        elif pooling == "mean_max":
            mean_emb = last_hidden.mean(dim=1)
            max_emb = last_hidden.max(dim=1).values
            embeddings = torch.cat([mean_emb, max_emb], dim=1)

        elif pooling == "cls_last4":
            # média das últimas 4 camadas do CLS
            last_four = hidden_states[-4:]
            stacked = torch.stack(last_four, dim=0)  # [4, batch, seq_len, hidden]
            embeddings = torch.mean(stacked, dim=0)[:, 0, :]

        else:
            raise ValueError("Pooling deve ser 'cls', 'mean', 'max', 'mean_max' ou 'cls_last4'")

    return embeddings.cpu().numpy()

'''
def get_embeddings(texts, tokenizer, model, device="cpu", pooling="cls"):
    inputs = tokenizer(texts, padding=True, truncation=True, max_length=128, return_tensors="pt")
    inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = model(**inputs)
        last_hidden = outputs.last_hidden_state  # [batch, seq_len, hidden]

        if pooling == "cls":
            embeddings = last_hidden[:, 0, :]  # pega [CLS]
        elif pooling == "mean":
            embeddings = last_hidden.mean(dim=1)  # média dos tokens
        else:
            raise ValueError("Pooling deve ser 'cls' ou 'mean'")

    return embeddings.cpu().numpy()
'''

texts = df["text"].tolist()
labels = df["label"].values

# Usar palavras do dicionário
X_words = df[list(lexicon.keys())]
vocab_size_words = X_words.shape[1]

# Normalizar palavras 
scaler = StandardScaler()
X_words_scaled = scaler.fit_transform(X_words)


X_embeddings = get_embeddings(texts, tokenizer, model, pooling="cls_last4")

# Concatenar embeddings + palavras
X_full = np.hstack([X_embeddings, X_words_scaled])
y = labels


skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
results = []

experiments_start_time = time.time()

for clf_name, clf in classifiers.items():
    print(f"\n=== {clf_name} ===")
    fold_metrics = []
    
    
    for train_idx, test_idx in skf.split(X_full, y):
        X_train, X_test = X_full[train_idx], X_full[test_idx]
        y_train, y_test = y[train_idx], y[test_idx]
        
        # Ajuste especial para MultinomialNB (precisa de entradas positivas)
        if clf_name == "MultinomialNB":
            from sklearn.preprocessing import MinMaxScaler
            scaler = MinMaxScaler()
            X_train = scaler.fit_transform(X_train)
            X_test = scaler.transform(X_test)
        
        clf.fit(X_train, y_train)
        y_pred = clf.predict(X_test)
        
        # Probabilidades ou scores para AUC
        if hasattr(clf, "predict_proba"):
            y_scores = clf.predict_proba(X_test)[:, 1]
        elif hasattr(clf, "decision_function"):
            y_scores = clf.decision_function(X_test)
        else:
            y_scores = None  # se o modelo não tiver, pula AUC

        # Algumas métricas
        acc = accuracy_score(y_test, y_pred)
        prec = precision_score(y_test, y_pred)
        rec = recall_score(y_test, y_pred)
        f1 = f1_score(y_test, y_pred)
        
        if y_scores is not None:
            auc = roc_auc_score(y_test, y_scores)
        else:
            auc = np.nan

        fold_metrics.append([acc, prec, rec, f1, auc])
    
    
    fold_metrics = np.array(fold_metrics, dtype=np.float64)
    mean_metrics = np.nanmean(fold_metrics, axis=0)  # ignora NaN se algum modelo não tiver AUC
    std_metrics = np.nanstd(fold_metrics, axis=0)
    vocab_size = X_full.shape[1]

    results.append([
        clf_name,
        *mean_metrics,
        *std_metrics,
        vocab_size
    ])
    
# Tempo total do experimento
experiment_time = time.time() - experiments_start_time

# Organiza resultados em DataFrame
results_df = pd.DataFrame(
    results, 
    columns=[
        "Modelo", 
        "Accuracy_mean", "Precision_mean", "Recall_mean", "F1_mean", "AUC_mean",
        "Accuracy_std", "Precision_std", "Recall_std", "F1_std", "AUC_std",
        "Vocab_size_combined"
    ]
)

results_df['total_experiment_time_sec'] = experiment_time


print(results_df)
results_df.to_csv('./resultados/results_FpOliveira_Sempre_processamento.csv', index=False)


=== LogisticRegression ===

=== BernoulliNB ===

=== MultinomialNB ===

=== LinearSVC ===

=== KNN ===

=== SGDClassifier ===

=== RandomForest ===

=== GradientBoosting ===

=== MLP ===
               Modelo  Accuracy_mean  Precision_mean  Recall_mean   F1_mean  \
0  LogisticRegression       0.882000        0.893202     0.868000  0.880245   
1         BernoulliNB       0.816667        0.850906     0.768000  0.807188   
2       MultinomialNB       0.821000        0.851262     0.778000  0.812842   
3           LinearSVC       0.863000        0.866226     0.858667  0.862376   
4                 KNN       0.837333        0.862471     0.802667  0.831390   
5       SGDClassifier       0.867000        0.886724     0.842000  0.863279   
6        RandomForest       0.841333        0.845245     0.836000  0.840473   
7    GradientBoosting       0.868000        0.875696     0.858000  0.866675   
8                 MLP       0.873667        0.879228     0.866667  0.872831   

   AUC_mean  Accuracy

### ruanchaves/bert-base-portuguese-cased-hatebr

In [17]:
# === Configurações ===

model_name = "ruanchaves/bert-base-portuguese-cased-hatebr"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModel.from_pretrained(
    model_name,
    use_safetensors=True # não usa toach.load
    )
model.eval()  # modo avaliação
def get_embeddings(texts, tokenizer, model, device="cpu", pooling="cls_last4"):
    inputs = tokenizer(
        texts, 
        padding=True, 
        truncation=True, 
        max_length=128, 
        return_tensors="pt"
    )
    inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = model(**inputs, output_hidden_states=True)
        hidden_states = outputs.hidden_states  # lista de camadas
        last_hidden = outputs.last_hidden_state  # [batch, seq_len, hidden]

        if pooling == "cls":
            embeddings = last_hidden[:, 0, :]

        elif pooling == "mean":
            embeddings = last_hidden.mean(dim=1)

        elif pooling == "max":
            embeddings = last_hidden.max(dim=1).values

        elif pooling == "mean_max":
            mean_emb = last_hidden.mean(dim=1)
            max_emb = last_hidden.max(dim=1).values
            embeddings = torch.cat([mean_emb, max_emb], dim=1)

        elif pooling == "cls_last4":
            # média das últimas 4 camadas do CLS
            last_four = hidden_states[-4:]
            stacked = torch.stack(last_four, dim=0)  # [4, batch, seq_len, hidden]
            embeddings = torch.mean(stacked, dim=0)[:, 0, :]

        else:
            raise ValueError("Pooling deve ser 'cls', 'mean', 'max', 'mean_max' ou 'cls_last4'")

    return embeddings.cpu().numpy()

'''
def get_embeddings(texts, tokenizer, model, device="cpu", pooling="cls"):
    inputs = tokenizer(texts, padding=True, truncation=True, max_length=128, return_tensors="pt")
    inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = model(**inputs)
        last_hidden = outputs.last_hidden_state  # [batch, seq_len, hidden]

        if pooling == "cls":
            embeddings = last_hidden[:, 0, :]  # pega [CLS]
        elif pooling == "mean":
            embeddings = last_hidden.mean(dim=1)  # média dos tokens
        else:
            raise ValueError("Pooling deve ser 'cls' ou 'mean'")

    return embeddings.cpu().numpy()
'''

texts = df["text"].tolist()
labels = df["label"].values

# Usar palavras do dicionário
X_words = df[list(lexicon.keys())]
vocab_size_words = X_words.shape[1]

# Normalizar palavras 
scaler = StandardScaler()
X_words_scaled = scaler.fit_transform(X_words)


X_embeddings = get_embeddings(texts, tokenizer, model, pooling="cls_last4")

# Concatenar embeddings + palavras
X_full = np.hstack([X_embeddings, X_words_scaled])
y = labels


skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
results = []

experiments_start_time = time.time()

for clf_name, clf in classifiers.items():
    print(f"\n=== {clf_name} ===")
    fold_metrics = []
    
    
    for train_idx, test_idx in skf.split(X_full, y):
        X_train, X_test = X_full[train_idx], X_full[test_idx]
        y_train, y_test = y[train_idx], y[test_idx]
        
        # Ajuste especial para MultinomialNB (precisa de entradas positivas)
        if clf_name == "MultinomialNB":
            from sklearn.preprocessing import MinMaxScaler
            scaler = MinMaxScaler()
            X_train = scaler.fit_transform(X_train)
            X_test = scaler.transform(X_test)
        
        clf.fit(X_train, y_train)
        y_pred = clf.predict(X_test)
        
        # Probabilidades ou scores para AUC
        if hasattr(clf, "predict_proba"):
            y_scores = clf.predict_proba(X_test)[:, 1]
        elif hasattr(clf, "decision_function"):
            y_scores = clf.decision_function(X_test)
        else:
            y_scores = None  # se o modelo não tiver, pula AUC

        # Algumas métricas
        acc = accuracy_score(y_test, y_pred)
        prec = precision_score(y_test, y_pred)
        rec = recall_score(y_test, y_pred)
        f1 = f1_score(y_test, y_pred)
        
        if y_scores is not None:
            auc = roc_auc_score(y_test, y_scores)
        else:
            auc = np.nan

        fold_metrics.append([acc, prec, rec, f1, auc])
    
    
    fold_metrics = np.array(fold_metrics, dtype=np.float64)
    mean_metrics = np.nanmean(fold_metrics, axis=0)  # ignora NaN se algum modelo não tiver AUC
    std_metrics = np.nanstd(fold_metrics, axis=0)
    vocab_size = X_full.shape[1]

    results.append([
        clf_name,
        *mean_metrics,
        *std_metrics,
        vocab_size
    ])
    
# Tempo total do experimento
experiment_time = time.time() - experiments_start_time

# Organiza resultados em DataFrame
results_df = pd.DataFrame(
    results, 
    columns=[
        "Modelo", 
        "Accuracy_mean", "Precision_mean", "Recall_mean", "F1_mean", "AUC_mean",
        "Accuracy_std", "Precision_std", "Recall_std", "F1_std", "AUC_std",
        "Vocab_size_combined"
    ]
)

results_df['total_experiment_time_sec'] = experiment_time

print(results_df)
results_df.to_csv('./resultados/results_ruanChaves_Sempre_processamento.csv', index=False)


=== LogisticRegression ===

=== BernoulliNB ===

=== MultinomialNB ===

=== LinearSVC ===

=== KNN ===

=== SGDClassifier ===

=== RandomForest ===

=== GradientBoosting ===

=== MLP ===
               Modelo  Accuracy_mean  Precision_mean  Recall_mean   F1_mean  \
0  LogisticRegression       0.869000        0.881580     0.852667  0.866687   
1         BernoulliNB       0.760667        0.741315     0.801333  0.769910   
2       MultinomialNB       0.761000        0.741457     0.802000  0.770331   
3           LinearSVC       0.865000        0.873154     0.854000  0.863398   
4                 KNN       0.811000        0.819813     0.797333  0.808345   
5       SGDClassifier       0.843333        0.890948     0.785333  0.832425   
6        RandomForest       0.802667        0.814470     0.784000  0.798894   
7    GradientBoosting       0.847333        0.857529     0.834000  0.845380   
8                 MLP       0.849000        0.857861     0.838000  0.846922   

   AUC_mean  Accuracy

## Modelo pré treinados - Com pré processamento

In [18]:
# === Pré-processamento dinâmico ===
df["text"] = preprocess_data(df, experiment="processed")  
df.head()

Pré-processamento ativado.


,text,label,num_tokens,aberração,aberração*,aberrações,aleij*,aleija,aleijada,aleijadas,...,progressista,progressistinha,reaça,revolucionário,socialista,sociopata,tucanhalha,vermelhinho,xenofóbico,zumbi
0,show clapping handsrrrrr light skin tone clap...,0,5,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,mais vez pt partir treva apelar mp proibir esc...,1,43,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,se puder aviserr quebrar vigíliar \n\n“pai est...,0,123,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,e esquerda tentar impedir trabalho,0,7,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,Deus perdoa porque saber fazer falar pronto as...,0,15,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


### melll-uff/bertweetbr

In [19]:

# === Configurações ===

model_name = "melll-uff/bertweetbr"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModel.from_pretrained(
    model_name,
    use_safetensors=True # não usa toach.load
    )
model.eval()  # modo avaliação

def get_embeddings(texts, tokenizer, model, device="cpu", pooling="cls_last4"):
    inputs = tokenizer(
        texts, 
        padding=True, 
        truncation=True, 
        max_length=128, 
        return_tensors="pt"
    )
    inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = model(**inputs, output_hidden_states=True)
        hidden_states = outputs.hidden_states  # lista de camadas
        last_hidden = outputs.last_hidden_state  # [batch, seq_len, hidden]

        if pooling == "cls":
            embeddings = last_hidden[:, 0, :]

        elif pooling == "mean":
            embeddings = last_hidden.mean(dim=1)

        elif pooling == "max":
            embeddings = last_hidden.max(dim=1).values

        elif pooling == "mean_max":
            mean_emb = last_hidden.mean(dim=1)
            max_emb = last_hidden.max(dim=1).values
            embeddings = torch.cat([mean_emb, max_emb], dim=1)

        elif pooling == "cls_last4":
            # média das últimas 4 camadas do CLS
            last_four = hidden_states[-4:]
            stacked = torch.stack(last_four, dim=0)  # [4, batch, seq_len, hidden]
            embeddings = torch.mean(stacked, dim=0)[:, 0, :]

        else:
            raise ValueError("Pooling deve ser 'cls', 'mean', 'max', 'mean_max' ou 'cls_last4'")

    return embeddings.cpu().numpy()

'''
def get_embeddings(texts, tokenizer, model, device="cpu", pooling="cls"):
    inputs = tokenizer(texts, padding=True, truncation=True, max_length=128, return_tensors="pt")
    inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = model(**inputs)
        last_hidden = outputs.last_hidden_state  # [batch, seq_len, hidden]

        if pooling == "cls":
            embeddings = last_hidden[:, 0, :]  # pega [CLS]
        elif pooling == "mean":
            embeddings = last_hidden.mean(dim=1)  # média dos tokens
        else:
            raise ValueError("Pooling deve ser 'cls' ou 'mean'")

    return embeddings.cpu().numpy()
'''

texts = df["text"].tolist()
labels = df["label"].values

# Usar palavras do dicionário
X_words = df[list(lexicon.keys())]
vocab_size_words = X_words.shape[1]

# Normalizar palavras 
scaler = StandardScaler()
X_words_scaled = scaler.fit_transform(X_words)


X_embeddings = get_embeddings(texts, tokenizer, model, pooling="cls_last4")

# Concatenar embeddings + palavras
X_full = np.hstack([X_embeddings, X_words_scaled])
y = labels



skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
results = []


experiments_start_time = time.time()

for clf_name, clf in classifiers.items():
    print(f"\n=== {clf_name} ===")
    fold_metrics = []
    
    
    for train_idx, test_idx in skf.split(X_full, y):
        X_train, X_test = X_full[train_idx], X_full[test_idx]
        y_train, y_test = y[train_idx], y[test_idx]
        
        # Ajuste especial para MultinomialNB (precisa de entradas positivas)
        if clf_name == "MultinomialNB":
            from sklearn.preprocessing import MinMaxScaler
            scaler = MinMaxScaler()
            X_train = scaler.fit_transform(X_train)
            X_test = scaler.transform(X_test)
        
        clf.fit(X_train, y_train)
        y_pred = clf.predict(X_test)
        
        # Probabilidades ou scores para AUC
        if hasattr(clf, "predict_proba"):
            y_scores = clf.predict_proba(X_test)[:, 1]
        elif hasattr(clf, "decision_function"):
            y_scores = clf.decision_function(X_test)
        else:
            y_scores = None  # se o modelo não tiver, pula AUC

        # Algumas métricas
        acc = accuracy_score(y_test, y_pred)
        prec = precision_score(y_test, y_pred)
        rec = recall_score(y_test, y_pred)
        f1 = f1_score(y_test, y_pred)
        
        if y_scores is not None:
            auc = roc_auc_score(y_test, y_scores)
        else:
            auc = np.nan

        fold_metrics.append([acc, prec, rec, f1, auc])
    
    
    fold_metrics = np.array(fold_metrics, dtype=np.float64)
    mean_metrics = np.nanmean(fold_metrics, axis=0)  # ignora NaN se algum modelo não tiver AUC
    std_metrics = np.nanstd(fold_metrics, axis=0)
    vocab_size = X_full.shape[1]

    results.append([
        clf_name,
        *mean_metrics,
        *std_metrics,
        vocab_size
    ])
    
# Tempo total do experimento
experiment_time = time.time() - experiments_start_time

# Organiza resultados em DataFrame
results_df = pd.DataFrame(
    results, 
    columns=[
        "Modelo", 
        "Accuracy_mean", "Precision_mean", "Recall_mean", "F1_mean", "AUC_mean",
        "Accuracy_std", "Precision_std", "Recall_std", "F1_std", "AUC_std",
        "Vocab_size_combined"
    ]
)

results_df['total_experiment_time_sec'] = experiment_time

print(results_df)
results_df.to_csv('./resultados/results_melll_uff_Compre_processamento.csv', index=False)

Some weights of RobertaModel were not initialized from the model checkpoint at melll-uff/bertweetbr and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.



=== LogisticRegression ===

=== BernoulliNB ===

=== MultinomialNB ===

=== LinearSVC ===

=== KNN ===

=== SGDClassifier ===

=== RandomForest ===

=== GradientBoosting ===

=== MLP ===
               Modelo  Accuracy_mean  Precision_mean  Recall_mean   F1_mean  \
0  LogisticRegression       0.867000        0.901513     0.824000  0.860971   
1         BernoulliNB       0.642667        0.636534     0.665333  0.650506   
2       MultinomialNB       0.760333        0.762903     0.756000  0.759358   
3           LinearSVC       0.865000        0.883133     0.841333  0.861676   
4                 KNN       0.808333        0.834578     0.769333  0.800502   
5       SGDClassifier       0.833667        0.824244     0.849333  0.836250   
6        RandomForest       0.784000        0.792752     0.770000  0.780937   
7    GradientBoosting       0.837333        0.850446     0.818667  0.834116   
8                 MLP       0.853000        0.856461     0.848000  0.852151   

   AUC_mean  Accuracy

### FpOliveira/tupi-bert-base-portuguese-cased

In [20]:
# === Configurações ===

model_name = "FpOliveira/tupi-bert-base-portuguese-cased"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModel.from_pretrained(
    model_name,
    use_safetensors=True # não usa toach.load
    )
model.eval()  # modo avaliação

def get_embeddings(texts, tokenizer, model, device="cpu", pooling="cls_last4"):
    inputs = tokenizer(
        texts, 
        padding=True, 
        truncation=True, 
        max_length=128, 
        return_tensors="pt"
    )
    inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = model(**inputs, output_hidden_states=True)
        hidden_states = outputs.hidden_states  # lista de camadas
        last_hidden = outputs.last_hidden_state  # [batch, seq_len, hidden]

        if pooling == "cls":
            embeddings = last_hidden[:, 0, :]

        elif pooling == "mean":
            embeddings = last_hidden.mean(dim=1)

        elif pooling == "max":
            embeddings = last_hidden.max(dim=1).values

        elif pooling == "mean_max":
            mean_emb = last_hidden.mean(dim=1)
            max_emb = last_hidden.max(dim=1).values
            embeddings = torch.cat([mean_emb, max_emb], dim=1)

        elif pooling == "cls_last4":
            # média das últimas 4 camadas do CLS
            last_four = hidden_states[-4:]
            stacked = torch.stack(last_four, dim=0)  # [4, batch, seq_len, hidden]
            embeddings = torch.mean(stacked, dim=0)[:, 0, :]

        else:
            raise ValueError("Pooling deve ser 'cls', 'mean', 'max', 'mean_max' ou 'cls_last4'")

    return embeddings.cpu().numpy()

'''
def get_embeddings(texts, tokenizer, model, device="cpu", pooling="cls"):
    inputs = tokenizer(texts, padding=True, truncation=True, max_length=128, return_tensors="pt")
    inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = model(**inputs)
        last_hidden = outputs.last_hidden_state  # [batch, seq_len, hidden]

        if pooling == "cls":
            embeddings = last_hidden[:, 0, :]  # pega [CLS]
        elif pooling == "mean":
            embeddings = last_hidden.mean(dim=1)  # média dos tokens
        else:
            raise ValueError("Pooling deve ser 'cls' ou 'mean'")

    return embeddings.cpu().numpy()
'''
texts = df["text"].tolist()
labels = df["label"].values

# Usar palavras do dicionário
X_words = df[list(lexicon.keys())]
vocab_size_words = X_words.shape[1]

# Normalizar palavras 
scaler = StandardScaler()
X_words_scaled = scaler.fit_transform(X_words)


X_embeddings = get_embeddings(texts, tokenizer, model, pooling="cls_last4")

# Concatenar embeddings + palavras
X_full = np.hstack([X_embeddings, X_words_scaled])
y = labels


skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
results = []


experiments_start_time = time.time()

for clf_name, clf in classifiers.items():
    print(f"\n=== {clf_name} ===")
    fold_metrics = []
    
    
    for train_idx, test_idx in skf.split(X_full, y):
        X_train, X_test = X_full[train_idx], X_full[test_idx]
        y_train, y_test = y[train_idx], y[test_idx]
        
        # Ajuste especial para MultinomialNB (precisa de entradas positivas)
        if clf_name == "MultinomialNB":
            from sklearn.preprocessing import MinMaxScaler
            scaler = MinMaxScaler()
            X_train = scaler.fit_transform(X_train)
            X_test = scaler.transform(X_test)
        
        clf.fit(X_train, y_train)
        y_pred = clf.predict(X_test)
        
        # Probabilidades ou scores para AUC
        if hasattr(clf, "predict_proba"):
            y_scores = clf.predict_proba(X_test)[:, 1]
        elif hasattr(clf, "decision_function"):
            y_scores = clf.decision_function(X_test)
        else:
            y_scores = None  # se o modelo não tiver, pula AUC

        # Algumas métricas
        acc = accuracy_score(y_test, y_pred)
        prec = precision_score(y_test, y_pred)
        rec = recall_score(y_test, y_pred)
        f1 = f1_score(y_test, y_pred)
        
        if y_scores is not None:
            auc = roc_auc_score(y_test, y_scores)
        else:
            auc = np.nan

        fold_metrics.append([acc, prec, rec, f1, auc])
    
    
    fold_metrics = np.array(fold_metrics, dtype=np.float64)
    mean_metrics = np.nanmean(fold_metrics, axis=0)  # ignora NaN se algum modelo não tiver AUC
    std_metrics = np.nanstd(fold_metrics, axis=0)
    vocab_size = X_full.shape[1]

    results.append([
        clf_name,
        *mean_metrics,
        *std_metrics,
        vocab_size
    ])

# Tempo total do experimento
experiment_time = time.time() - experiments_start_time

# Organiza resultados em DataFrame
results_df = pd.DataFrame(
    results, 
    columns=[
        "Modelo", 
        "Accuracy_mean", "Precision_mean", "Recall_mean", "F1_mean", "AUC_mean",
        "Accuracy_std", "Precision_std", "Recall_std", "F1_std", "AUC_std",
        "Vocab_size_combined"
    ]
)

results_df['total_experiment_time_sec'] = experiment_time

print(results_df)
results_df.to_csv('./resultados/results_FpOliveira_Compre_processamento.csv', index=False)


=== LogisticRegression ===

=== BernoulliNB ===

=== MultinomialNB ===

=== LinearSVC ===

=== KNN ===

=== SGDClassifier ===

=== RandomForest ===

=== GradientBoosting ===

=== MLP ===
               Modelo  Accuracy_mean  Precision_mean  Recall_mean   F1_mean  \
0  LogisticRegression       0.865667        0.882445     0.844000  0.862629   
1         BernoulliNB       0.753333        0.817567     0.652667  0.725771   
2       MultinomialNB       0.762333        0.819914     0.672667  0.738904   
3           LinearSVC       0.853000        0.864119     0.838667  0.850952   
4                 KNN       0.829667        0.860109     0.787333  0.822033   
5       SGDClassifier       0.851333        0.858224     0.842667  0.849804   
6        RandomForest       0.798333        0.800014     0.796000  0.797905   
7    GradientBoosting       0.846333        0.849980     0.841333  0.845456   
8                 MLP       0.854333        0.864939     0.840000  0.852126   

   AUC_mean  Accuracy

### ruanchaves/bert-base-portuguese-cased-hatebr

In [21]:
# === Configurações ===

model_name = "ruanchaves/bert-base-portuguese-cased-hatebr"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModel.from_pretrained(
    model_name,
    use_safetensors=True # não usa toach.load
    )
model.eval()  # modo avaliação

def get_embeddings(texts, tokenizer, model, device="cpu", pooling="cls_last4"):
    inputs = tokenizer(
        texts, 
        padding=True, 
        truncation=True, 
        max_length=128, 
        return_tensors="pt"
    )
    inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = model(**inputs, output_hidden_states=True)
        hidden_states = outputs.hidden_states  # lista de camadas
        last_hidden = outputs.last_hidden_state  # [batch, seq_len, hidden]

        if pooling == "cls":
            embeddings = last_hidden[:, 0, :]

        elif pooling == "mean":
            embeddings = last_hidden.mean(dim=1)

        elif pooling == "max":
            embeddings = last_hidden.max(dim=1).values

        elif pooling == "mean_max":
            mean_emb = last_hidden.mean(dim=1)
            max_emb = last_hidden.max(dim=1).values
            embeddings = torch.cat([mean_emb, max_emb], dim=1)

        elif pooling == "cls_last4":
            # média das últimas 4 camadas do CLS
            last_four = hidden_states[-4:]
            stacked = torch.stack(last_four, dim=0)  # [4, batch, seq_len, hidden]
            embeddings = torch.mean(stacked, dim=0)[:, 0, :]

        else:
            raise ValueError("Pooling deve ser 'cls', 'mean', 'max', 'mean_max' ou 'cls_last4'")

    return embeddings.cpu().numpy()


'''
def get_embeddings(texts, tokenizer, model, device="cpu", pooling="cls"):
    inputs = tokenizer(texts, padding=True, truncation=True, max_length=128, return_tensors="pt")
    inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = model(**inputs)
        last_hidden = outputs.last_hidden_state  # [batch, seq_len, hidden]

        if pooling == "cls":
            embeddings = last_hidden[:, 0, :]  # pega [CLS]
        elif pooling == "mean":
            embeddings = last_hidden.mean(dim=1)  # média dos tokens
        else:
            raise ValueError("Pooling deve ser 'cls' ou 'mean'")

    return embeddings.cpu().numpy()
'''
texts = df["text"].tolist()
labels = df["label"].values


# Usar palavras do dicionário
X_words = df[list(lexicon.keys())]
vocab_size_words = X_words.shape[1]

# Normalizar palavras 
scaler = StandardScaler()
X_words_scaled = scaler.fit_transform(X_words)


X_embeddings = get_embeddings(texts, tokenizer, model, pooling="cls_last4")

# Concatenar embeddings + palavras
X_full = np.hstack([X_embeddings, X_words_scaled])
y = labels


skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
results = []


experiments_start_time = time.time()

for clf_name, clf in classifiers.items():
    print(f"\n=== {clf_name} ===")
    fold_metrics = []
    
    
    for train_idx, test_idx in skf.split(X_full, y):
        X_train, X_test = X_full[train_idx], X_full[test_idx]
        y_train, y_test = y[train_idx], y[test_idx]
        
        # Ajuste especial para MultinomialNB (precisa de entradas positivas)
        if clf_name == "MultinomialNB":
            from sklearn.preprocessing import MinMaxScaler
            scaler = MinMaxScaler()
            X_train = scaler.fit_transform(X_train)
            X_test = scaler.transform(X_test)
        
        clf.fit(X_train, y_train)
        y_pred = clf.predict(X_test)
        
        # Probabilidades ou scores para AUC
        if hasattr(clf, "predict_proba"):
            y_scores = clf.predict_proba(X_test)[:, 1]
        elif hasattr(clf, "decision_function"):
            y_scores = clf.decision_function(X_test)
        else:
            y_scores = None  # se o modelo não tiver, pula AUC

        # Algumas métricas
        acc = accuracy_score(y_test, y_pred)
        prec = precision_score(y_test, y_pred)
        rec = recall_score(y_test, y_pred)
        f1 = f1_score(y_test, y_pred)
        
        if y_scores is not None:
            auc = roc_auc_score(y_test, y_scores)
        else:
            auc = np.nan

        fold_metrics.append([acc, prec, rec, f1, auc])
    
    
    fold_metrics = np.array(fold_metrics, dtype=np.float64)
    mean_metrics = np.nanmean(fold_metrics, axis=0)  # ignora NaN se algum modelo não tiver AUC
    std_metrics = np.nanstd(fold_metrics, axis=0)
    vocab_size = X_full.shape[1]

    results.append([
        clf_name,
        *mean_metrics,
        *std_metrics,
        vocab_size
    ])

# Tempo total do experimento
experiment_time = time.time() - experiments_start_time

# Organiza resultados em DataFrame
results_df = pd.DataFrame(
    results, 
    columns=[
        "Modelo", 
        "Accuracy_mean", "Precision_mean", "Recall_mean", "F1_mean", "AUC_mean",
        "Accuracy_std", "Precision_std", "Recall_std", "F1_std", "AUC_std",
        "Vocab_size_combined"
    ]
)

results_df['total_experiment_time_sec'] = experiment_time

print(results_df)
results_df.to_csv('./resultados/results_ruanChaves_Compre_processamento.csv', index=False)


=== LogisticRegression ===

=== BernoulliNB ===

=== MultinomialNB ===

=== LinearSVC ===

=== KNN ===

=== SGDClassifier ===

=== RandomForest ===

=== GradientBoosting ===

=== MLP ===
               Modelo  Accuracy_mean  Precision_mean  Recall_mean   F1_mean  \
0  LogisticRegression       0.857667        0.877700     0.831333  0.853772   
1         BernoulliNB       0.737333        0.707453     0.810000  0.754869   
2       MultinomialNB       0.738667        0.706736     0.816667  0.757349   
3           LinearSVC       0.853667        0.862986     0.841333  0.851827   
4                 KNN       0.817667        0.822774     0.810000  0.816210   
5       SGDClassifier       0.810000        0.774961     0.890000  0.825617   
6        RandomForest       0.793000        0.814488     0.759333  0.785675   
7    GradientBoosting       0.835000        0.850385     0.813333  0.831363   
8                 MLP       0.850000        0.866764     0.827333  0.846379   

   AUC_mean  Accuracy

## Analisando os erros do melhor resultados

In [12]:
from sklearn.preprocessing import MinMaxScaler

model_name = "FpOliveira/tupi-bert-base-portuguese-cased"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModel.from_pretrained(model_name, use_safetensors=True)
model.eval()

def get_embeddings(texts, tokenizer, model, device="cpu"):
    inputs = tokenizer(
        texts,
        padding=True,
        truncation=True,
        max_length=128,
        return_tensors="pt"
    )
    inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = model(**inputs, output_hidden_states=True)
        hidden_states = outputs.hidden_states
        last_four = hidden_states[-4:]
        stacked = torch.stack(last_four)
        embeddings = torch.mean(stacked, dim=0)[:, 0, :]

    return embeddings.cpu().numpy()


def build_lexicon_features(texts, lexicon_words):
    X = np.zeros((len(texts), len(lexicon_words)))

    for i, text in enumerate(texts):
        text = text.lower()
        for j, word in enumerate(lexicon_words):
            if word and re.search(rf"\b{re.escape(word)}", text):
                X[i, j] = 1  # presença (ou use += 1 para contagem)
    return X

texts = df["text"].tolist()
labels = df["label"].values

lexicon_words = [w.replace("*", "").lower() for w in lexicon.keys()]

# Features do dicionário
#X_words = df[list(lexicon.keys())]
X_words = build_lexicon_features(texts, lexicon_words)

scaler = StandardScaler()
X_words_scaled = scaler.fit_transform(X_words)

# Embeddings
X_embeddings = get_embeddings(texts, tokenizer, model)

# ===== Léxico =====
lexicon_words = [w.replace("*", "").lower() for w in lexicon.keys()]
X_words = build_lexicon_features(texts, lexicon_words)

scaler = StandardScaler()
X_words_scaled = scaler.fit_transform(X_words)

# Combinação
X_full = np.hstack([X_embeddings, X_words_scaled])
y = labels

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

all_errors = []
results = []

start_time = time.time()

for clf_name, clf in classifiers.items():
    fold_metrics = []

    for fold, (train_idx, test_idx) in enumerate(skf.split(X_full, y)):
        X_train, X_test = X_full[train_idx], X_full[test_idx]
        y_train, y_test = y[train_idx], y[test_idx]

        if clf_name == "MultinomialNB":
            scaler_nb = MinMaxScaler()
            X_train = scaler_nb.fit_transform(X_train)
            X_test = scaler_nb.transform(X_test)

        clf.fit(X_train, y_train)
        y_pred = clf.predict(X_test)

        if hasattr(clf, "predict_proba"):
            y_scores = clf.predict_proba(X_test)[:, 1]
        elif hasattr(clf, "decision_function"):
            y_scores = clf.decision_function(X_test)
        else:
            y_scores = None

        # Métricas
        fold_metrics.append([
            accuracy_score(y_test, y_pred),
            precision_score(y_test, y_pred),
            recall_score(y_test, y_pred),
            f1_score(y_test, y_pred),
            roc_auc_score(y_test, y_scores) if y_scores is not None else np.nan
        ])

        # 🔍 Coletar erros
        for i, idx in enumerate(test_idx):
            if y_pred[i] != y_test[i]:
                error_type = "FP" if y_pred[i] == 1 else "FN"
                all_errors.append({
                    "modelo": clf_name,
                    "fold": fold,
                    "texto": texts[idx],
                    "y_true": y_test[i],
                    "y_pred": y_pred[i],
                    "tipo_erro": error_type,
                    "score": y_scores[i] if y_scores is not None else None
                })

    fold_metrics = np.array(fold_metrics)
    results.append([
        clf_name,
        *fold_metrics.mean(axis=0),
        *fold_metrics.std(axis=0),
        X_full.shape[1]
    ])

total_time = time.time() - start_time

results_df = pd.DataFrame(
    results,
    columns=[
        "Modelo",
        "Accuracy_mean", "Precision_mean", "Recall_mean", "F1_mean", "AUC_mean",
        "Accuracy_std", "Precision_std", "Recall_std", "F1_std", "AUC_std",
        "Vocab_size"
    ]
)

results_df["tempo_total_segundos"] = total_time
results_df.to_csv("resultados_modelos.csv", index=False)

errors_df = pd.DataFrame(all_errors)

# Separar erros
false_positives = errors_df[errors_df["tipo_erro"] == "FP"]
false_negatives = errors_df[errors_df["tipo_erro"] == "FN"]

false_positives.to_csv("erros_falsos_positivos.csv", index=False)
false_negatives.to_csv("erros_falsos_negativos.csv", index=False)

print("Falsos Positivos:", len(false_positives))
print("Falsos Negativos:", len(false_negatives))


  File "d:\Arquivos_Acer\Documents\UFC\PrejudicePT-br-main\ambiente_prejudice\lib\site-packages\joblib\externals\loky\backend\context.py", line 257, in _count_physical_cores
    cpu_info = subprocess.run(
  File "C:\Users\Melissa Felipe\AppData\Local\Programs\Python\Python310\lib\subprocess.py", line 503, in run
    with Popen(*popenargs, **kwargs) as process:
  File "C:\Users\Melissa Felipe\AppData\Local\Programs\Python\Python310\lib\subprocess.py", line 971, in __init__
    self._execute_child(args, executable, preexec_fn, close_fds,
  File "C:\Users\Melissa Felipe\AppData\Local\Programs\Python\Python310\lib\subprocess.py", line 1456, in _execute_child
    hp, ht, pid, tid = _winapi.CreateProcess(executable, args,


Falsos Positivos: 1804
Falsos Negativos: 2521


In [17]:
# palavras mais comuns no falso negativos
from sklearn.feature_extraction.text import CountVectorizer

fn_texts = false_negatives["texto"]

vectorizer = CountVectorizer(max_features=50)
X_fn = vectorizer.fit_transform(fn_texts)

top_fn = zip(
    vectorizer.get_feature_names_out(),
    X_fn.sum(axis=0).A1
)

sorted(top_fn, key=lambda x: x[1], reverse=True)


[('que', 2938),
 ('de', 2894),
 ('não', 1566),
 ('do', 1558),
 ('para', 1026),
 ('brazil', 1017),
 ('com', 991),
 ('um', 906),
 ('no', 883),
 ('em', 874),
 ('da', 869),
 ('se', 817),
 ('os', 732),
 ('as', 602),
 ('deus', 579),
 ('na', 579),
 ('bolsonaro', 572),
 ('uma', 542),
 ('lula', 518),
 ('por', 513),
 ('ele', 471),
 ('como', 460),
 ('rolling_on_the_floor_laughing', 444),
 ('esse', 434),
 ('brasil', 420),
 ('vai', 411),
 ('está', 405),
 ('tem', 402),
 ('isso', 350),
 ('mas', 349),
 ('quem', 345),
 ('eu', 343),
 ('todos', 342),
 ('pra', 339),
 ('mais', 336),
 ('esquerda', 328),
 ('dia', 322),
 ('ser', 320),
 ('só', 310),
 ('ao', 288),
 ('thumbs_up', 280),
 ('foi', 274),
 ('nosso', 273),
 ('vamos', 273),
 ('dos', 271),
 ('são', 271),
 ('essa', 260),
 ('muito', 259),
 ('luladrao', 254),
 ('ou', 246)]

In [18]:
false_negatives.sample(5)[["texto", "y_true", "y_pred"]]


,texto,y_true,y_pred
4030,Oi estou vendo que vc acha que me sentiria ofe...,1,0
1330,Imbecil \nPensa pra falar!\nO Lula deveria ter...,1,0
1230,TALVEZ NÃO VAI TER NADA DIA TRINTA SO VA...,1,0
1673,É difícil ter comunhão com um crente desse que...,1,0
2862,Bolsonaro pede a Deus que nós não experimentem...,1,0


In [16]:
# palavras mais comuns no falso positivos


fn_texts = false_positives["texto"]

vectorizer = CountVectorizer(max_features=50)
X_fn = vectorizer.fit_transform(fn_texts)

top_fn = zip(
    vectorizer.get_feature_names_out(),
    X_fn.sum(axis=0).A1
)

sorted(top_fn, key=lambda x: x[1], reverse=True)


[('de', 1903),
 ('que', 1773),
 ('não', 1082),
 ('do', 946),
 ('para', 614),
 ('se', 578),
 ('um', 571),
 ('brazil', 561),
 ('da', 556),
 ('com', 550),
 ('em', 519),
 ('no', 515),
 ('deus', 462),
 ('os', 447),
 ('na', 420),
 ('esquerda', 406),
 ('por', 382),
 ('uma', 380),
 ('tem', 342),
 ('bolsonaro', 341),
 ('mais', 327),
 ('só', 287),
 ('brasil', 282),
 ('vai', 262),
 ('eu', 234),
 ('ser', 234),
 ('as', 232),
 ('mas', 231),
 ('quem', 223),
 ('pra', 219),
 ('presidente', 218),
 ('ao', 214),
 ('todos', 213),
 ('ele', 205),
 ('como', 204),
 ('lula', 197),
 ('essa', 182),
 ('são', 181),
 ('esse', 180),
 ('dos', 178),
 ('nosso', 178),
 ('contra', 176),
 ('isso', 173),
 ('nos', 171),
 ('eles', 165),
 ('money_bag', 160),
 ('vamos', 160),
 ('foi', 159),
 ('rolling_on_the_floor_laughing', 159),
 ('tudo', 153)]